## 1. Conexão com o banco


In [ ]:
import pandas as pd
import os
import plotly.graph_objects as go
import threading
import time
from dash import Dash, html, dcc, Input, Output, callback, State
from dash.exceptions import PreventUpdate
from sqlmodel import create_engine

In [ ]:
_NAVY      = "#1B3A6B"
_NAVY_DARK = "#142D55"
_WHITE     = "#FFFFFF"
_SURFACE   = "#F4F6F9"
_BORDER    = "#DDE3EC"
_TEXT_MAIN = "#1A1A2E"
_TEXT_MUT  = "#6B7280"
_FONT = "Inter, system-ui, -apple-system, sans-serif"


def setor_maior_ch(df):
    """Retorna o setor com a maior carga horária total.

    Args:
        df: DataFrame com coluna "Setor" (str) e "CH" (float).

    Returns:
        Nome do setor (str) com o maior somatório de CH.
    """
    return df.groupby("Setor")["CH"].sum().idxmax()


COR_DEMANDA_STYLES = {
    "alta":  {"color": "#C0392B", "fontWeight": "500", "fontSize": "13px"},
    "média": {"color": "#D68910", "fontWeight": "500", "fontSize": "13px"},
    "baixa": {"color": "#1E8449", "fontWeight": "500", "fontSize": "13px"},
}

CORES_RDYLGN = [
    "#a50026", "#b81a26", "#cc3327", "#d94f2a", "#e66b2e",
    "#f08933", "#f5a640", "#f9c24e", "#f5b800", "#e8a800",
    "#d4960a", "#bfb825", "#a8c43a", "#8dc44e", "#72be5a",
    "#5ab947", "#3daa3d", "#229a38", "#0e8a2f", "#027a28",
    "#006a23", "#005a1d", "#004a17", "#003a11", "#002a0b",
]

In [ ]:
def cria_cabecalho() -> html.Div:
    """Monta o cabeçalho do painel lateral.

    Exibe um ícone estilizado com a letra "C" e o texto
    "Centro de Tecnologia / Gestão de professores".

    Returns:
        html.Div com o cabeçalho do painel lateral.
    """
    return html.Div(
        id="cabecalho",
        style={
            "display": "flex",
            "alignItems": "center",
            "gap": "12px",
            "padding": "20px 16px 16px 16px",
            "borderBottom": f"1px solid {_NAVY_DARK}",
            "marginBottom": "8px",
        },
        children=[
            html.Div(
                "C",
                style={
                    "width": "40px",
                    "height": "40px",
                    "borderRadius": "8px",
                    "background": _WHITE,
                    "color": _NAVY,
                    "display": "flex",
                    "alignItems": "center",
                    "justifyContent": "center",
                    "fontFamily": _FONT,
                    "fontWeight": "700",
                    "fontSize": "20px",
                    "flexShrink": "0",
                },
            ),
            html.Div(
                children=[
                    html.Div(
                        "Centro de Tecnologia",
                        style={
                            "fontFamily": _FONT,
                            "fontWeight": "600",
                            "fontSize": "14px",
                            "color": _WHITE,
                            "lineHeight": "1.2",
                        },
                    ),
                    html.Div(
                        "Gestão de professores",
                        style={
                            "fontFamily": _FONT,
                            "fontWeight": "400",
                            "fontSize": "11px",
                            "color": "rgba(255,255,255,0.65)",
                            "marginTop": "2px",
                        },
                    ),
                ]
            ),
        ],
    )


def cria_painel_lateral(df_organizado, lista_ofertas):
    """Constrói o painel lateral de controles interativos.

    Cria dropdown de seleção de setor, botão de comparação entre
    setores e range slider para filtro temporal de períodos.

    Args:
        df_organizado: DataFrame consolidado com dados de ofertas.
        lista_ofertas: Lista ordenada de períodos disponíveis.

    Returns:
        html.Div com os controles do painel lateral.
    """
    n = len(lista_ofertas) - 1

    marks = {
        0: {
            "label": str(lista_ofertas[0]),
            "style": {"color": "rgba(255,255,255,0.7)", "fontSize": "9px", "fontWeight": "500"},
        },
        n: {
            "label": str(lista_ofertas[-1]),
            "style": {
                "color": "rgba(255,255,255,0.7)",
                "fontSize": "9px",
                "fontWeight": "500",
                "whiteSpace": "nowrap",
            },
        },
    }

    _label_style = {
        "fontFamily": _FONT,
        "fontSize": "11px",
        "fontWeight": "500",
        "padding": "6px 2px",
        "color": "rgba(255,255,255,0.55)",
        "textTransform": "uppercase",
        "letterSpacing": "0.08em",
        "marginBottom": "6px",
        "marginTop": "10px",
    }

    _dropdown_style = {
        "borderRadius": "6px",
        "fontSize": "13px",
        "fontFamily": _FONT,
    }

    setores_disponiveis = sorted(df_organizado["Setor"].unique())
    setor_inicial = setor_maior_ch(df_organizado)

    return html.Div(
        id="painel-lateral-controles",
        style={"padding": "0 8px 24px 8px"},
        children=[

            html.Div("Setor", style=_label_style),
            html.Div(
                style={"position": "relative"},
                children=[
                    html.Span(
                        "\U0001f50d",
                        style={
                            "position": "absolute",
                            "left": "10px",
                            "top": "50%",
                            "transform": "translateY(-50%)",
                            "fontSize": "12px",
                            "zIndex": "10",
                            "pointerEvents": "none",
                            "opacity": "0.5",
                        },
                    ),
                    dcc.Dropdown(
                        id="barra-pesquisa-1",
                        options=[{"label": s, "value": s} for s in setores_disponiveis],
                        value=None,
                        placeholder="Selecionar Setor",
                        searchable=True,
                        clearable=False,
                        style={**_dropdown_style, "paddingLeft": "28px"},
                    ),
                ],
            ),

            html.Button(
                " Comparar Setores",
                id="botao-comparar",
                n_clicks=0,
                style={
                    "marginTop": "16px",
                    "width": "100%",
                    "height": "34px",
                    "padding": "8px 12px",
                    "fontFamily": _FONT,
                    "fontSize": "13px",
                    "fontWeight": "600",
                    "color": _WHITE,
                    "background": "rgba(255,255,255,0.15)",
                    "border": "1px solid rgba(255,255,255,0.25)",
                    "borderRadius": "6px",
                    "cursor": "pointer",
                    "textAlign": "center",
                },
            ),

            html.Div(
                id="container-barra-pesquisa-2",
                style={"display": "none"},
                children=[
                    html.Div("Comparar com", style={**_label_style, "marginTop": "6px"}),
                    html.Div(
                        style={"position": "relative"},
                        children=[
                            html.Span(
                                "\U0001f50d",
                                style={
                                    "position": "absolute",
                                    "left": "10px",
                                    "top": "50%",
                                    "transform": "translateY(-50%)",
                                    "fontSize": "12px",
                                    "zIndex": "10",
                                    "pointerEvents": "none",
                                    "opacity": "0.5",
                                },
                            ),
                            dcc.Dropdown(
                                id="barra-pesquisa-2",
                                options=[{"label": s, "value": s} for s in setores_disponiveis],
                                placeholder="Selecionar Setor",
                                searchable=True,
                                clearable=False,
                                style={**_dropdown_style, "paddingLeft": "28px"},
                            ),
                        ],
                    ),
                ],
            ),

            html.Div("Período", style=_label_style),

            html.Div(
                style={
                    "display": "flex",
                    "gap": "8px",
                    "marginBottom": "12px",
                },
                children=[
                    html.Div(
                        id="label-slider-inicio",
                        children=str(lista_ofertas[0]),
                        style={
                            "flex": "1",
                            "textAlign": "center",
                            "background": _WHITE,
                            "color": _NAVY,
                            "fontFamily": _FONT,
                            "fontWeight": "600",
                            "fontSize": "13px",
                            "padding": "5px 4px",
                            "borderRadius": "6px",
                        },
                    ),
                    html.Div(
                        id="label-slider-fim",
                        children=str(lista_ofertas[-1]),
                        style={
                            "flex": "1",
                            "textAlign": "center",
                            "background": _WHITE,
                            "color": _NAVY,
                            "fontFamily": _FONT,
                            "fontWeight": "600",
                            "fontSize": "13px",
                            "padding": "5px 4px",
                            "borderRadius": "6px",
                        },
                    ),
                ],
            ),

            html.Div(
                style={"marginLeft": "6px", "marginRight": "6px"},
                children=[
                    dcc.RangeSlider(
                        id="range-slider",
                        min=0,
                        max=n,
                        step=1,
                        value=[0, n],
                        marks=marks,
                        tooltip={"always_visible": False},
                    ),
                ],
            ),
        ],
    )


def cria_container_grafico(df_organizado):
    """Cria o container que envolve o gráfico de CH por setor.

    Args:
        df_organizado: DataFrame consolidado para alimentar o gráfico.

    Returns:
        html.Div com o gráfico de barras encapsulado.
    """
    setores_disponiveis = sorted(df_organizado["Setor"].unique())
    setor_inicial = setor_maior_ch(df_organizado)
    return html.Div(
        style={
            "background": _WHITE,
            "borderRadius": "10px",
            "border": f"1px solid {_BORDER}",
            "overflow": "visible",
            "height": "400px",
            "paddingTop": "12px",
        },
        children=[
            dcc.Graph(
                id="grafico",
                figure=cria_grafico(df_organizado, [setor_inicial]),
                config={"displayModeBar": False, "responsive": True},
                style={"height": "100%", "width": "100%"},
            )
        ],
    )


def cria_container_tabelas() -> html.Div:
    """Cria o container para exibição das tabelas de professores.

    No modo simples exibe uma tabela; no modo comparação exibe duas.
    O preenchimento é feito via callback.

    Returns:
        html.Div com espaço para uma ou duas tabelas.
    """
    return html.Div(
        style={"display": "flex", "flexDirection": "column", "gap": "16px"},
        children=[

            html.Div(id="tabela-professores-1"),

            html.Div(
                id="container-comparacao",
                style={"display": "none"},
                children=[
                    html.Div(id="tabela-professores-2"),
                ],
            ),
        ],
    )


def cria_layout(df_organizado, lista_ofertas):
    """Monta o layout completo do dashboard.

    Estrutura o painel lateral (controles) à esquerda e a área
    principal (gráfico + tabelas) à direita.

    Args:
        df_organizado: DataFrame consolidado para estado inicial.
        lista_ofertas: Lista de períodos para o range slider.

    Returns:
        html.Div com o layout completo do dashboard.
    """
    setores_disponiveis = sorted(df_organizado["Setor"].unique())
    setor_inicial = setor_maior_ch(df_organizado)
    return html.Div(
        id="container-geral",
        style={
            "display": "flex",
            "minHeight": "100vh",
            "fontFamily": _FONT,
            "background": _SURFACE,
        },
        children=[

            dcc.Store(
                id="store-df-filtrado",
                data=df_organizado.to_dict("records")
            ),
            dcc.Store(
                id="store-comparacao",
                data=False
            ),

            html.Div(
                id="painel-lateral",
                style={
                    "width": "250px",
                    "minWidth": "250px",
                    "background": _NAVY,
                    "display": "flex",
                    "flexDirection": "column",
                    "minHeight": "100vh",
                },
                children=[
                    cria_cabecalho(),
                    cria_painel_lateral(df_organizado, lista_ofertas),
                ],
            ),

            html.Div(
                id="conteudo-principal",
                style={
                    "flex": "1",
                    "display": "flex",
                    "flexDirection": "column",
                    "padding": "24px",
                    "gap": "20px",
                    "overflowY": "auto",
                },
                children=[

                    html.Div(
                        id="div-superior",
                        style={
                            "display": "flex",
                            "gap": "24px",
                            "alignItems": "flex-start",
                        },
                        children=[

                            html.Div(
                                id="titulo-data",
                                style={"minWidth": "220px"},
                                children=[
                                    html.H1(
                                        "Setores de atuação",
                                        style={
                                            "fontFamily": _FONT,
                                            "fontWeight": "700",
                                            "fontSize": "26px",
                                            "color": _TEXT_MAIN,
                                            "margin": "0 0 4px 0",
                                            "lineHeight": "1.2",
                                        },
                                    ),
                                    html.Div(
                                        id="subtitulo-data",
                                        style={
                                            "fontFamily": _FONT,
                                            "fontSize": "13px",
                                            "color": _TEXT_MUT,
                                            "fontWeight": "400",
                                        },
                                    ),
                                ],
                            ),

                            html.Div(
                                style={"flex": "1"},
                                children=[cria_container_grafico(df_organizado)],
                            ),
                        ],
                    ),

                    html.Div(
                        id="div-inferior",
                        children=[cria_container_tabelas()],
                    ),
                ],
            ),
        ],
    )

## 2. Consultas


In [ ]:
BINS_DEMANDA   = [-float("inf"), 8, 12, float("inf")]
LABELS_DEMANDA = ["Baixa", "Média", "Alta"]


def carrega_dados(path: str) -> pd.DataFrame:
    """
    Conecta-se a um banco de dados SQLite, extrai os dados das tabelas
    'setores', 'disciplinas', 'professor' e 'oferta', realiza uma limpeza
    na coluna 'turma' (mantendo apenas os dígitos numéricos como inteiros)
    e combina todas as informações.

    Retorna df_bruto com todas as colunas brutas resultantes dos joins,
    sem filtragem. A seleção e limpeza de colunas ocorre em organiza_dados().
    """
    engine = create_engine(f"sqlite:///{path}")
    df_setores = pd.read_sql_query("SELECT * FROM setores", con=engine)
    df_disciplinas = pd.read_sql_query("SELECT * FROM disciplinas", con=engine)
    df_professores = pd.read_sql_query("SELECT * FROM professor", con=engine)
    df_ofertas = pd.read_sql_query("SELECT * FROM oferta", con=engine)
    if df_ofertas["turma"].dtype == object:
        df_ofertas["turma"] = df_ofertas["turma"].str.extract(r"(\d+)").astype(int)
    df_bruto = (df_ofertas.merge(df_professores.rename(columns={"nome_professor": "nome_professor1"}),
                                 on="id_professor", how="left")
                          .merge(
              df_professores.rename(columns={"id_professor": "id_professor2", "nome_professor": "nome_professor2"}),
              on="id_professor2", how="left")
                          .merge(df_disciplinas, on="codigo", how="left")
                          .merge(df_setores, on="id_setor", how="left"))
    return df_bruto


def organiza_dados(df_bruto: pd.DataFrame, remover_a_definir: bool, SEMANAS_PERIODO: int = 15) -> pd.DataFrame:
    """
    Recebe o df_bruto, organiza seus dados de maneira tal que facilite a alimentação
    do Dashboard e as operações que serão efetuadas. Retorna o df_organizado.

    Colunas do df retornado:
    - Oferta: período da oferta (ex: 2026.1)
    - Professor: nome do professor (uma linha por professor x disciplina)
    - Setor: nome do setor
    - Disciplina: nome da disciplina
    - CH: média de horas semanais do professor naquela disciplina
          (CH_periodo / SEMANAS_PERIODO)
    - Demanda: Alta (CH > 12), Média (8 < CH ≤ 12), Baixa (CH ≤ 8)
    """
    df = df_bruto.copy()
    df = df.drop(columns=[
        "id_oferta", "id_professor", "id_professor2", "turma", "horario",
        "local", "matriculados", "capacidade", "id_setor"], errors="ignore")
    df_p1 = df[df["nome_professor1"].notna()].copy()
    df_p1["Professor"] = df_p1["nome_professor1"]
    df_p1["CH_periodo"] = df_p1["ch_professor"]
    df_p2 = df[df["nome_professor2"].notna()].copy()
    df_p2["Professor"] = df_p2["nome_professor2"]
    df_p2["CH_periodo"] = df_p2["ch_professor2"]
    df_long = pd.concat([df_p1, df_p2], ignore_index=True)
    if remover_a_definir:
        df_long = df_long[df_long["Professor"] != "A DEFINIR"]
    df_long["CH"] = (df_long["CH_periodo"] / SEMANAS_PERIODO).round(2)
    df_long["Demanda"] = pd.cut(
        df_long["CH"],
        bins=BINS_DEMANDA,
        labels=LABELS_DEMANDA
    )
    df_organizado = df_long.rename(columns={
        "nome_setor": "Setor",
        "nome_disciplina": "Disciplina",
        "oferta": "Oferta",
    })[[
        "Oferta", "Professor", "Setor", "Disciplina", "CH", "Demanda"
    ]]
    return df_organizado.reset_index(drop=True)


def agrega_por_professor(df_organizado: pd.DataFrame, setor: str) -> pd.DataFrame:
    """
    Filtra os professores que atuam no setor informado. A coluna Disciplinas lista
    apenas as disciplinas do professor naquele setor, concatenadas por ", ".
    CH_setor é a carga horária do professor apenas naquele setor; CH é a carga
    horária total em todos os setores. A Demanda é calculada sobre a CH total,
    usando os mesmos bins definidos em BINS_DEMANDA / LABELS_DEMANDA.
    Retorna o df_ptabela.
    """
    professores_do_setor = df_organizado.loc[
        df_organizado["Setor"] == setor, "Professor"
    ].unique()
    df_disciplinas_setor = (
        df_organizado[
            (df_organizado["Professor"].isin(professores_do_setor)) &
            (df_organizado["Setor"] == setor)
        ]
        .groupby("Professor", as_index=False)
        .agg(Disciplinas=("Disciplina", lambda x: ", ".join(x)))
    )
    df_ch_setor = (
        df_organizado[
            (df_organizado["Professor"].isin(professores_do_setor)) &
            (df_organizado["Setor"] == setor)
        ]
        .groupby("Professor", as_index=False)
        .agg(CH_setor=("CH", "sum"))
    )
    df_ch_total = (
        df_organizado[df_organizado["Professor"].isin(professores_do_setor)]
        .groupby("Professor", as_index=False)
        .agg(CH=("CH", "sum"))
    )
    df_agg = df_disciplinas_setor.merge(df_ch_setor, on="Professor").merge(df_ch_total, on="Professor")
    df_agg["Setor"] = setor
    df_agg["Demanda"] = pd.cut(
        df_agg["CH"].round(2),
        bins=BINS_DEMANDA,
        labels=LABELS_DEMANDA
    )
    df_ptabela = df_agg[[
        "Professor", "Setor", "Disciplinas", "CH_setor", "CH", "Demanda"
    ]].reset_index(drop=True)
    return df_ptabela


def identifica_ofertas(df_bruto: pd.DataFrame) -> list:
    """
    Recebe o df_bruto e identifica os semestres ofertados para alimentar
    o Range Slider de seleção temporal, retornando a lista_ofertas.
    """
    lista_ofertas = sorted(df_bruto['oferta'].unique().tolist())
    return lista_ofertas



def seleciona_dados(df_organizado: pd.DataFrame, semestres_selecionados: list) -> pd.DataFrame:
    """
    Filtra o df_organizado mantendo apenas as linhas cujo semestre esteja dentro
    do intervalo temporal estabelecido pelo Range Slider.

    semestres_selecionados: lista com dois elementos [idx_inicio, idx_fim]
    correspondendo aos índices da lista de ofertas retornada por
    identifica_ofertas(). O filtro mantém as linhas cujo semestre esteja
    no intervalo lista_ofertas[idx_inicio : idx_fim + 1].
    Retorna df_filtrado.
    """
    lista_ofertas = sorted(df_organizado["Oferta"].unique().tolist())
    idx_inicio, idx_fim = semestres_selecionados
    ofertas_validas = lista_ofertas[idx_inicio : idx_fim + 1]
    df_filtrado = df_organizado[df_organizado["Oferta"].isin(ofertas_validas)].copy()
    return df_filtrado.reset_index(drop = True)


def gera_opcoes_dropdown_1(df_filtrado: pd.DataFrame, texto_digitado_1: str) -> list:
    """
    Filtra os setores disponíveis com base no texto que o usuário está digitando
    em barra-pesquisa-1, retornando as sugestões para o dropdown (lista_dropdown_1)
    em tempo real.
    """
    setores = sorted(df_filtrado["Setor"].unique().tolist())
    if not texto_digitado_1:
        return setores
    lista_dropdown_1 = [s for s in setores if texto_digitado_1.lower() in s.lower()]
    return lista_dropdown_1


def gera_opcoes_dropdown_2(df_filtrado: pd.DataFrame, texto_digitado_2: str, setor_selecionado_1: str) -> list:
    """
    Filtra os setores pelo texto digitado em barra-pesquisa-2, excluindo o setor
    já selecionado em barra-pesquisa-1. Retorna lista_dropdown_2.
    """
    setores = sorted(df_filtrado["Setor"].unique().tolist())
    if setor_selecionado_1:
        setores = [s for s in setores if s != setor_selecionado_1]
    if not texto_digitado_2:
        return setores
    lista_dropdown_2 = [s for s in setores if texto_digitado_2.lower() in s.lower()]
    return lista_dropdown_2

## 3. Construção dos gráficos


In [ ]:
def cria_tabela_professores(df_filtrado: pd.DataFrame, setores: list) -> html.Div:
    """
    Filtra os dados para os setores selecionados e apresenta a distribuicao de
    professores por disciplina em forma de tabela HTML estilizada.
    """
    LABELS = {
        "Professor": "Professor",
        "Setor": "Setor",
        "Disciplinas": "Disciplina",
        "CH_setor": "CH Setor",
        "CH": "CH Total",
        "Demanda": "Demanda",
    }
    def _cor_demanda(nivel: str) -> dict:
        """Retorna o estilo CSS para a badge de demanda.

        Args:
            nivel: String com o nível ("Alta", "Média" ou "Baixa").

        Returns:
            Dict com propriedades CSS (color, fontWeight, fontSize).
    """
        return COR_DEMANDA_STYLES.get(nivel.lower().strip(), {"color": "#888", "fontWeight": "500", "fontSize": "13px"})
    def _monta_tabela(setor: str) -> html.Div:
        """Constrói a tabela HTML de professores para um setor.

        Agrega os dados por professor, formata CH e demanda,
        e retorna a tabela estilizada.

        Args:
            setor: Nome do setor a ser filtrado.

        Returns:
            html.Div com a tabela de distribuição por professor.
    """
        df = agrega_por_professor(df_filtrado, setor)
        cabecalho = html.Thead(
            html.Tr([
                html.Th(LABELS.get(col, col), style={
                    "padding": "10px 16px",
                    "textAlign": "left",
                    "fontSize": "13px",
                    "fontWeight": "700",
                    "color": "#444",
                    "borderBottom": "1px solid #ddd",
                    "background": "#f0f0f0",
                })
                for col in df.columns
            ])
        )
        linhas = []
        for _, row in df.iterrows():
            celulas = []
            for col in df.columns:
                valor = row[col]
                if col == "Demanda":
                    celula = html.Td(
                        html.Span(str(valor).capitalize(), style=_cor_demanda(str(valor))),
                        style={"padding": "12px 26px", "verticalAlign": "middle"},
                    )
                elif col == "CH_setor":
                    celula = html.Td(
                        html.Span(f"{valor:.1f}h", style={
                            "background": "#E8F4FD",
                            "color": "#1A5276",
                            "fontWeight": "600",
                            "fontSize": "12px",
                            "padding": "3px 8px",
                            "borderRadius": "4px",
                        }),
                        style={"padding": "12px 26px", "verticalAlign": "middle"},
                    )
                elif col == "CH":
                    celula = html.Td(
                        f"{valor:.1f}h",
                        style={
                            "padding": "12px 16px",
                            "fontSize": "13px",
                            "color": "#333",
                            "verticalAlign": "middle",
                        },
                    )
                else:
                    celula = html.Td(
                        str(valor),
                        style={
                            "padding": "12px 16px",
                            "fontSize": "13px",
                            "color": "#333",
                            "verticalAlign": "middle",
                        },
                    )
                celulas.append(celula)
            linhas.append(html.Tr(celulas, style={"borderBottom": "0.5px solid #eee"}))
        tabela = html.Table(
            [cabecalho, html.Tbody(linhas)],
            style={"width": "100%", "borderCollapse": "collapse", "fontSize": "13px"},
        )
        return html.Div(
            style={
                "background": "#f7f7f7",
                "border": "1px solid #ddd",
                "borderRadius": "8px",
                "overflow": "hidden",
                "marginBottom": "24px",
            },
            children=[
                html.Div(
                    f"Distribuição por professor em {setor}",
                    style={
                        "padding": "14px 20px",
                        "fontSize": "15px",
                        "fontWeight": "500",
                        "color": "#222",
                        "borderBottom": "1px solid #ddd",
                        "background": "#efefef",
                    },
                ),
                tabela,
            ],
        )
    if len(setores) == 2:
        return html.Div([
            _monta_tabela(setores[0]),
            _monta_tabela(setores[1]),
        ])
    else:
        return html.Div([
            _monta_tabela(setores[0]),
        ])


def cria_grafico(df_filtrado: pd.DataFrame, setores_selecionados: list) -> go.Figure:
    """
    Gera um grafico de barras da Carga Horaria Semanal Total por Setor.
    """

    df_agg = (
        df_filtrado
        .groupby("Setor", as_index=False)
        .agg(
            CH_total=("CH", "sum"),
            Disciplinas=("Disciplina", lambda x: "<br>".join(x.dropna().unique()))
        )
        .sort_values("CH_total", ascending=False)
        .reset_index(drop=True)
    )
    n = len(df_agg)
    indices = [round(i * (len(CORES_RDYLGN) - 1) / max(n - 1, 1)) for i in range(n)]
    cores_barras = [CORES_RDYLGN[i] for i in indices]
    selecionados = set(s.strip() for s in (setores_selecionados or []))
    contorno_cores = [
        "#1A5276" if s in selecionados else "rgba(0,0,0,0)"
        for s in df_agg["Setor"]
    ]
    contorno_widths = [
        6 if s in selecionados else 0
        for s in df_agg["Setor"]
    ]
    customdata = list(zip(df_agg["Setor"], df_agg["CH_total"], df_agg["Disciplinas"]))
    fig = go.Figure(
        go.Bar(
            x=df_agg["Setor"].tolist(),
            y=df_agg["CH_total"].tolist(),
            marker=dict(
                color=cores_barras,
                opacity=[0.6 if s in selecionados else 1.0 for s in df_agg["Setor"]],
                line=dict(
                    color=contorno_cores,
                    width=contorno_widths,
                ),
            ),
            customdata=customdata,
            hovertemplate=(
                "<b>Setor:</b> %{customdata[0]}<br>"
                "<b>CH Total:</b> %{customdata[1]:.1f}h/sem<br>"
                "<b>Disciplinas:</b><br>%{customdata[2]}"
                "<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title=dict(
            text="Carga Horaria Semanal Total por Setor",
            x=0.5,
            y=0.95,
            xanchor="center",
            font=dict(size=16, color="#222", family="Arial"),
        ),
        xaxis=dict(
            title="Setor",
            showticklabels=False,
            showgrid=False,
            zeroline=False,
        ),
        yaxis=dict(
            title="CH Total (Horas/Semana)",
            tickfont=dict(size=11, color="#444"),
            showgrid=True,
            gridcolor="#e8e8e8",
            zeroline=False,
        ),
        plot_bgcolor="#ffffff",
        paper_bgcolor="#ffffff",
        margin=dict(t=25, b=20, l=35, r=10),
        hoverlabel=dict(
            bgcolor="#ffffff",
            bordercolor="#cccccc",
            font=dict(size=12, color="#333"),
            align="left"
        ),
        hovermode="x",
        bargap=0.15,
        showlegend=False,
    )
    return fig

In [ ]:
def criar_app():
    """Inicializa o dashboard completo.

    Carrega os dados do banco SQLite, cria a aplicação Dash,
    registra todos os callbacks de interação e retorna o objeto app.

    Returns:
        app: Objeto Dash pronto para execução.
    """
    import os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DB_PATH = str(ROOT / "dados" / "banco" / "ofertas.db")
    REMOVER_A_DEFINIR = False

    df_bruto = carrega_dados(DB_PATH)
    df_organizado = organiza_dados(df_bruto, REMOVER_A_DEFINIR)
    lista_ofertas = identifica_ofertas(df_bruto)

    app = Dash(__name__, suppress_callback_exceptions=True)
    app.layout = cria_layout(df_organizado, lista_ofertas)

    @app.callback(
        Output("barra-pesquisa-1", "value"),
        Input("store-df-filtrado", "data")
    )
    def definir_setor_inicial(data):
        """Define o setor inicial no dropdown ao carregar os dados.

        Acionado quando o store-df-filtrado é populado pela primeira vez.
        Seleciona automaticamente o setor com maior CH total.

        Args:
            data: Lista de dicionários do store (ou None).

        Returns:
            Nome do setor (str) com maior CH, ou PreventUpdate se vazio.
        """
        if not data:
            raise PreventUpdate
        df_filtrado = pd.DataFrame(data)
        return setor_maior_ch(df_filtrado)


    @app.callback(
        Output("store-df-filtrado", "data"),
        Input("range-slider", "value")
    )
    def atualizar_store(range_tempo):
        """Filtra o DataFrame pelo intervalo temporal e atualiza o store.

        Acionado pelo RangeSlider. Converte o DataFrame filtrado
        para o formato de dicionários usado no dcc.Store.

        Args:
            range_tempo: Lista [idx_inicio, idx_fim] com índices das ofertas.

        Returns:
            Lista de dicionários (records) para o store-df-filtrado.
        """
        df_filtrado = seleciona_dados(df_organizado, range_tempo)
        return df_filtrado.to_dict("records")

    @app.callback(
        Output("grafico", "figure"),
        Output("tabela-professores-1", "children"),
        Output("tabela-professores-2", "children"),
        Output("subtitulo-data", "children"),
        Input("store-df-filtrado", "data"),
        Input("barra-pesquisa-1", "value"),
        Input("barra-pesquisa-2", "value"),
        Input("store-comparacao", "data")
    )
    def atualizar_dashboard(data, setor_1, setor_2, modo_comparacao):
        """Atualiza gráfico, tabelas e subtítulo do dashboard.

        Função central do dashboard: reage à seleção de setor(es),
        filtro temporal e modo de comparação, regenerando o gráfico
        de CH por setor e as tabelas de distribuição de professores.

        Args:
            data: Dados do store-df-filtrado (lista de dicts).
            setor_1: Setor selecionado no primeiro dropdown.
            setor_2: Setor selecionado no segundo dropdown (se modo comparação).
            modo_comparacao: Booleano indicando se o modo comparação está ativo.

        Returns:
            Tuple (figure, tabela_1, tabela_2, subtitulo) para atualizar
            os outputs do dashboard.
        """
        if not data or not setor_1:
            return go.Figure(), html.Div(), html.Div(), ""
        df_filtrado = pd.DataFrame(data)
        setores_selecionados = [setor_1]
        if modo_comparacao and setor_2:
            setores_selecionados.append(setor_2)
        fig = cria_grafico(df_filtrado, setores_selecionados)
        ofertas = sorted(df_filtrado["Oferta"].unique())
        subtitulo = f"Dados referentes aos períodos: {ofertas[0]} a {ofertas[-1]}" if ofertas else ""
        tabela_1 = cria_tabela_professores(df_filtrado, [setor_1])
        tabela_2 = cria_tabela_professores(df_filtrado, [setor_2]) if (modo_comparacao and setor_2) else html.Div()
        return fig, tabela_1, tabela_2, subtitulo

    @app.callback(
        Output("barra-pesquisa-1", "options"),
        Output("barra-pesquisa-2", "options"),
        Input("store-df-filtrado", "data"),
        Input("barra-pesquisa-1", "search_value"),
        Input("barra-pesquisa-2", "search_value"),
        State("barra-pesquisa-1", "value")
    )
    def atualizar_dropdowns(data, busca_1, busca_2, setor_1):
        """Filtra as opções dos dropdowns conforme o texto digitado.

        Realiza busca textual nos setores disponíveis, excluindo
        o setor já selecionado no dropdown 1 das opções do dropdown 2.

        Args:
            data: Dados do store-df-filtrado.
            busca_1: Texto digitado no dropdown 1.
            busca_2: Texto digitado no dropdown 2.
            setor_1: Setor atualmente selecionado no dropdown 1.

        Returns:
            Tuple (opcoes_1, opcoes_2) com listas de setores filtradas.
        """
        df_filtrado = pd.DataFrame(data) if data else pd.DataFrame()
        opcoes_1 = gera_opcoes_dropdown_1(df_filtrado, busca_1 or "")
        setor_excluido = setor_1 or ""
        opcoes_2 = gera_opcoes_dropdown_2(df_filtrado, busca_2 or "", setor_excluido)
        return opcoes_1, opcoes_2

    @app.callback(
        Output("store-comparacao", "data"),
        Input("botao-comparar", "n_clicks"),
        State("store-comparacao", "data"),
        prevent_initial_call=True
    )
    def toggle_comparacao(n_clicks, ativo):
        """Alterna o modo de comparação entre setores.

        Inverte o estado atual do store-comparação a cada clique
        no botão "Comparar Setores".

        Args:
            n_clicks: Número de cliques no botão.
            ativo: Estado atual do modo comparação.

        Returns:
            Booleano com o novo estado (True/False).
        """
        return not ativo

    @app.callback(
        Output("container-comparacao", "style"),
        Output("container-barra-pesquisa-2", "style"),
        Input("store-comparacao", "data")
    )
    def exibir_segunda_tabela(modo_comparacao):
        """Controla a visibilidade da segunda tabela e do segundo dropdown.

        Quando o modo comparação está ativo, exibe os elementos;
        caso contrário, oculta-os.

        Args:
            modo_comparacao: Booleano indicando se o modo está ativo.

        Returns:
            Tuple com dois dicts de estilo CSS (display: block ou none).
        """
        display = "block" if modo_comparacao else "none"
        return {"display": display}, {"display": display}
    return app


## 4. Inicialização do Dashboard


In [ ]:
import logging
logging.getLogger("werkzeug").setLevel(logging.ERROR)

app = criar_app()

if __name__ == "__main__":
    app.run(debug=True, port=8050)


Dash is running on http://127.0.0.1:38063/



INFO:dash.dash:Dash is running on http://127.0.0.1:38063/



 * Serving Flask app '__main__'
 * Debug mode: on

           DASHBOARD ONLINE

        LINK:  https://orange-parks-say.loca.lt

---------------------------------------------------------
  SE APARECER UMA TELA DE CONFIRMACAO:
  1. Copie o IP mostrado na tela (ex: 34.143.246.30)
  2. Cole no campo 'IP Address'
  3. Clique em 'Continue'
  (Isso acontece apenas 1 vez a cada 7 dias)
---------------------------------------------------------

  Ctrl+C para encerrar

